# 🎙️ 播客总结 — 第 3 周任务

**作者：** 维克多征服者

该笔记本采用播客音频文件 (MP3)，使用 HuggingFace 的 Whisper 模型对其进行转录，
然后使用 Meta 的 Llama 3.2（量化为 4 位）生成结构化播客摘要。

**一切仅在 HuggingFace 模型上运行 - 无需 OpenAI API！**

### 你将学到什么：
- 如何使用 HuggingFace `pipeline()` 进行音频转录 (Whisper)
- 如何加载具有 4 位量化的大型语言模型 (BitsAndBytes)
- 如何使用“apply_chat_template()”进行结构化提示
- 如何使用“TextStreamer”逐个流式传输 LLM 输出

### 要求：
- 使用 **T4 GPU** 在 Google Colab 上运行（免费套餐有效！）
- 可以访问 Llama 3.2 的 HuggingFace 帐户
- 上传到 Google Drive 的 MP3 播客文件

## 步骤0：安装依赖项

我们需要特定版本的“transformers”、“bitsandbytes”（用于 4 位量化）、
和“加速”（用于跨设备智能模型加载）。

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

## 第 1 步：导入

HuggingFace 关键导入说明：
- `AutoTokenizer` — 转换文本↔数字（标记），以便模型可以理解它
- `AutoModelForCausalLM` — 加载文本生成模型（如 Llama）
- `TextStreamer` — 在生成令牌时打印令牌（实时流式传输！）
- `BitsAndBytesConfig` — 配置 4 位量化以适应小型 GPU 中的大型模型
- `pipeline` — 一种高级助手，它将模型 + 分词器 + 预处理捆绑到一个调用中

In [ ]:
import os
import torch
from IPython.display import Markdown, display
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline,
)

## 步骤 2：常数和设置

我们定义要使用的模型：
- **Whisper** (`openai/whisper-medium.en`) — 托管在 HuggingFace 上的开源语音转文本模型
- **Llama 3.2** (`meta-llama/Llama-3.2-3B-Instruct`) — Meta 的 30 亿参数指令调整模型

In [ ]:
# 型号
# Models
WHISPER_MODEL = "openai/whisper-medium.en"
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

## 步骤 3：连接 Google Drive 并加载音频

安装您的 Google 云端硬盘，以便笔记本可以访问您的播客文件。

**说明：**
1. 在 Google 云端硬盘中创建一个名为“llms”的文件夹
2. 将 MP3 播客文件上传到该文件夹中
3. 使用您的文件名更新下面的“audio_filename”

In [ ]:
# 挂载 Google 云端硬盘
# Mount Google Drive
drive.mount("/content/drive")

# 指向您的播客文件 - 更改文件名以匹配您的！
# Point to your podcast file — change the filename to match yours!
audio_filename = "/content/drive/MyDrive/llms/podcast_extract.mp3"

# 验证文件是否存在
# Verify the file exists
if os.path.exists(audio_filename):
    file_size_mb = os.path.getsize(audio_filename) / (1024 * 1024)
    print(f"✅ Found audio file: {audio_filename}")
    print(f"📁 File size: {file_size_mb:.1f} MB")
else:
    print(f"❌ File not found: {audio_filename}")
    print("Please upload your podcast MP3 to Google Drive in the 'llms' folder.")

## 步骤 4：登录 HuggingFace Hub

Llama 3.2 是一个**门控模型** — 您需要：
1. 前往 [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) 并请求访问权限
2. 在 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 创建 HuggingFace 令牌
3. 将其添加到 Colab Secrets（左侧边栏中的🔑图标）作为“HF_TOKEN”

In [ ]:
# 使用 Colab Secrets 的令牌登录 HuggingFace
# Login to HuggingFace using your token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("✅ Logged in to HuggingFace Hub!")

---

## 🎧 第 1 部分：使用 Whisper 进行转录

### 什么是`pipeline()`？

将“pipeline()”视为**单行快捷方式**：
1. 从 HuggingFace Hub 下载模型
2. 下载匹配的分词器/处理器
3. 设置所有预处理（音频 → 梅尔频谱图 → 标记）
4. 运行推理
5. 将输出后处理回文本

如果没有 `pipeline()`，您将需要大约 20 行代码来手动完成所有这些工作！

In [ ]:
# 创建转录管道
# Create the transcription pipeline
# - 型号：使用哪种 Whisper 型号
# - model: which Whisper model to use
# - dtype: float16 uses half the memory of float32
# - 设备：'cuda' = 使用 GPU
# - device: 'cuda' = use the GPU
# - return_timestamps：在输出中包含计时信息
# - return_timestamps: include timing info in the output

whisper_pipe = pipeline(
    task="automatic-speech-recognition",
    model=WHISPER_MODEL,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True,
)

print("✅ Whisper pipeline loaded!")

In [ ]:
# 运行转录——只需传递文件路径即可！
# Run the transcription — just pass the file path!
print("🎙️ Transcribing podcast... (this may take a minute)\n")

result = whisper_pipe(audio_filename)
transcription = result["text"]

print("=" * 60)
print("📝 TRANSCRIPTION RESULT:")
print("=" * 60)
print(transcription)
print(f"\n📊 Word count: {len(transcription.split())}")

---

## 🧠 第 2 部分：用 Llama 3.2 进行总结

现在，我们将转录内容输入 Llama 3.2 以生成结构化摘要。

### 什么是量化？

Llama 3.2 3B 通常需要约 6 GB 的 GPU 内存（以 float16 表示）。通过 **4 位量化**，
我们将模型权重从每个参数 16 位压缩为 4 位，其中：
- **将内存使用量减少约 4 倍**（适用于免费的 Colab T4！）
- 由于采用 NF4（Normal Float 4）格式，**几乎不影响质量**

`BitsAndBytesConfig` 控制这种压缩。

In [ ]:
# 配置 4 位量化
# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load weights in 4-bit format
    bnb_4bit_use_double_quant=True,  # Extra compression on the quantization constants
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use bfloat16 for computations (good balance)
    bnb_4bit_quant_type="nf4",      # Normal Float 4 — optimized for neural networks
)

print("✅ Quantization config ready!")

### 加载分词器和模型

**Tokenizer** = 人类文本和数字之间的转换器。
- `"Hello world"` → `[15043, 1917]` (编码)
- `[15043, 1917]` → `"Hello world"`（解码）

**模型** = 生成新令牌的实际神经网络。
- 获取一系列令牌 ID
- 预测下一个标记，一次一个
- `device_map="auto"` 根据需要自动将模型层放置在 GPU/CPU 上

In [ ]:
# 加载分词器
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
tokenizer.pad_token = tokenizer.eos_token  # Required for Llama

# 使用 4 位量化加载模型
# Load the model with 4-bit quantization
print("⏳ Loading Llama 3.2 (4-bit quantized)... this takes about 1-2 minutes...")
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    device_map="auto",
    quantization_config=quant_config,
)

print("✅ Model loaded!")
print(f"📊 Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

### 构建提示

我们使用具有两个角色的**聊天模板**：
- `system` — 告诉模型如何行为（它的身份和规则）
- `user` — 提供实际任务（要总结的转录）

`apply_chat_template()` 将其转换为 Llama 期望的确切令牌格式，
包括特殊标记，如`<|begin_of_text|>`、`<|start_header_id|>`等。

In [ ]:
# 定义系统提示符——告诉Llama我们想要什么样的输出
# Define the system prompt — tells Llama what kind of output we want
system_message = """
You are an expert podcast summarizer. Given a transcript of a podcast episode,
produce a well-structured summary in markdown (without code blocks) that includes:

1. **Episode Overview** — A 2-3 sentence high-level summary
2. **Key Topics Discussed** — Bullet points of the main subjects covered
3. **Notable Quotes** — Any memorable or impactful statements (in quotes)
4. **Key Takeaways** — The most important insights a listener should remember
5. **Who Should Listen** — What kind of audience would benefit from this episode

Keep it concise but comprehensive. Use a friendly, engaging tone.
"""

# 构建嵌入转录的用户提示
# Build the user prompt with the transcription embedded
user_prompt = f"""
Below is a transcript from a podcast episode.
Please create a structured summary following the format specified.

Transcription:
{transcription}
"""

# 以聊天格式组装消息
# Assemble the messages in chat format
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]

print("✅ Prompt ready!")
print(f"📊 System prompt: {len(system_message.split())} words")
print(f"📊 User prompt: {len(user_prompt.split())} words (includes transcription)")

### 生成摘要（使用流式传输！）

**TextStreamer** 在生成时打印每个标记 - 因此您可以看到输出
实时逐字显示，就像 ChatGPT 一样！

In [ ]:
# 使用聊天模板将消息转换为令牌 ID
# Convert messages to token IDs using the chat template
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# 设置实时流媒体
# Set up real-time streaming
streamer = TextStreamer(tokenizer)

# 产生！
# Generate!
print("🧠 Generating podcast summary...\n")
print("=" * 60)

outputs = model.generate(
    inputs,
    max_new_tokens=2000,  # Maximum length of the summary
    streamer=streamer,    # Stream tokens as they're generated
)

print("=" * 60)
print("\n✅ Summary generation complete!")

### 显示摘要（渲染的 Markdown）

In [ ]:
# 解码完整输出并显示为格式化 Markdown
# Decode the full output and display as formatted markdown
full_response = tokenizer.decode(outputs[0])

# 仅提取助理的响应（跳过提示）
# Extract only the assistant's response (skip the prompt)
# Llama 使用 <|start_header_id|>assistant<|end_header_id|> 来标记响应
# Llama uses <|start_header_id|>assistant<|end_header_id|> to mark the response
if "assistant" in full_response:
    summary = full_response.split("assistant")[-1]
    # 清理所有剩余的特殊标记
    # Clean up any remaining special tokens
    summary = summary.replace("<|end_header_id|>", "")
    summary = summary.replace("<|eot_id|>", "")
    summary = summary.replace("<|begin_of_text|>", "")
    summary = summary.strip()
else:
    summary = full_response

display(Markdown("# 🎙️ Podcast Summary\n\n" + summary))

---

## 🔍 第 3 部分：比较转录与摘要

让我们并排看看原始转录和经过修改的摘要。

In [ ]:
display(Markdown("## 📝 Raw Transcription\n\n" + transcription))
print("\n" + "=" * 60 + "\n")
display(Markdown("## 🎯 AI-Generated Summary\n\n" + summary))

---

## 📊 奖励：播客统计

In [ ]:
transcript_words = len(transcription.split())
summary_words = len(summary.split())
compression = (1 - summary_words / transcript_words) * 100 if transcript_words > 0 else 0

stats = f"""
# # 📊 播客统计
# # 📊 Podcast Stats

| Metric | Value |
|--------|-------|
| Transcription words | {transcript_words:,} |
| Summary words | {summary_words:,} |
| Compression ratio | {compression:.1f}% reduction |
| Whisper model | {WHISPER_MODEL} |
| LLM model | {LLAMA_MODEL} |
| Quantization | 4-bit NF4 |
| Model memory | {model.get_memory_footprint() / 1e9:.2f} GB |
"""

display(Markdown(stats))

---

## 🎓 你学到了什么（HuggingFace 概念）

### HuggingFace 生态系统 — 备忘单

|概念 |它有什么作用 |用于此笔记本|
|---------|-------------|----------------------|
| **HugingFace 中心** |托管超过 50 万个模型的平台——类似于 GitHub，但针对 AI 模型 |下载低语和骆驼|
| **`管道()`** |一行快捷方式：下载模型 + 分词器，运行推理 |耳语转录 |
| **`自动标记器`** |文本 ↔ 令牌 ID 转换器（每个模型都有自己的！） |将提示转换为 Llama 代币 |
| **`AutoModelForCausalLM`** |加载文本生成模型 |正在加载骆驼 3.2 |
| **`BitsAndBytesConfig`** | 4 位量化 — 缩小模型以适应小型 GPU |在 T4 中拟合 3B 参数 |
| **`TextStreamer`** |打印生成的令牌（实时输出）|实时摘要生成 |
| **`apply_chat_template()`** |将消息格式化为特定于模型的令牌格式 |构建 Llama 提示符 |
| **`device_map="自动"`** |自动在 GPU/CPU 之间分割模型 |智能内存管理|

### 流程
```
Audio File (MP3)
    ↓ pipeline("automatic-speech-recognition")
Raw Text (Transcription)
    ↓ apply_chat_template() + tokenizer
Token IDs
    ↓ model.generate() with TextStreamer
Structured Summary (Markdown)
```